In [2]:
# W terminalu VS Code zainstaluj:
# python -m pip install xgboost pandas scikit-learn matplotlib

import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb

from sklearn.metrics import accuracy_score, roc_curve, roc_auc_score
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV

print("Start programu...")

# =========================================================
# 1. Wczytanie danych
# =========================================================
df = pd.read_csv("f1_processed_dataset.csv")
print("Plik CSV wczytany poprawnie.")
print("Kształt danych:", df.shape)

# =========================================================
# 2. Usunięcie leakage
# =========================================================
# positionOrder to wynik końcowy wyścigu, więc nie wolno go używać
if "positionOrder" in df.columns:
    df = df.drop(columns=["positionOrder"])

# =========================================================
# 3. Kodowanie zmiennych tekstowych
# =========================================================
target_col = "top3"

if target_col not in df.columns:
    raise ValueError(f"Brakuje kolumny docelowej: {target_col}")

df = df.dropna().copy()

label_encoders = {}
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

if target_col in categorical_cols:
    categorical_cols.remove(target_col)

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

print("Zakodowane kolumny:", categorical_cols)

# =========================================================
# 4. Losowy podział: 80% train, 10% validation, 10% test
# =========================================================
train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df[target_col]
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df[target_col]
)

print("Dane treningowe:", train_df.shape)
print("Dane walidacyjne:", val_df.shape)
print("Dane testowe:", test_df.shape)

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col].astype(int)

X_val = val_df.drop(columns=[target_col])
y_val = val_df[target_col].astype(int)

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col].astype(int)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("Rozkład klas w treningu:")
print(y_train.value_counts())
print("Rozkład klas w walidacji:")
print(y_val.value_counts())
print("Rozkład klas w teście:")
print(y_test.value_counts())

# =========================================================
# 5. Bazowy model XGBoost
# =========================================================
xgb_model = xgb.XGBClassifier(
    objective="binary:logistic",
    random_state=42,
    n_estimators=100,
    eval_metric="logloss"
)

xgb_model.fit(X_train, y_train)
print("Model bazowy wytrenowany.")

# =========================================================
# 6. Ocena na zbiorze walidacyjnym
# =========================================================
y_val_pred = xgb_model.predict(X_val)
y_val_prob = xgb_model.predict_proba(X_val)[:, 1]

val_accuracy = accuracy_score(y_val, y_val_pred)
val_auc = roc_auc_score(y_val, y_val_prob)

print("Dokładność na zbiorze walidacyjnym:", val_accuracy)
print("AUC na zbiorze walidacyjnym:", val_auc)

# =========================================================
# 7. Walidacja krzyżowa na zbiorze treningowym
# =========================================================
scores_ = cross_val_score(
    xgb_model, X_train, y_train, cv=5, scoring="accuracy"
)
print("Dokładności predykcji (CV):", scores_)
print("Średnia dokładność (CV):", scores_.mean())

# =========================================================
# 8. Grid Search dla n_estimators
# =========================================================
param_grid = {
    "n_estimators": [50, 100, 200, 500]
}

grid_search = GridSearchCV(
    estimator=xgb.XGBClassifier(
        objective="binary:logistic",
        random_state=42,
        eval_metric="logloss"
    ),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
print("Najlepsze parametry:", grid_search.best_params_)

# =========================================================
# 9. Ocena najlepszego modelu na zbiorze testowym
# =========================================================
y_test_pred = best_model.predict(X_test)
y_test_prob = best_model.predict_proba(X_test)[:, 1]

test_accuracy = accuracy_score(y_test, y_test_pred)
test_auc = roc_auc_score(y_test, y_test_prob)

print("Dokładność predykcji na zbiorze testowym:", test_accuracy)
print("AUC na zbiorze testowym:", test_auc)

# =========================================================
# 10. Krzywa ROC
# =========================================================
fpr, tpr, thresholds = roc_curve(y_test, y_test_prob)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label="XGBoost")
plt.plot([0, 1], [0, 1], "--", label="Losowy klasyfikator")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Krzywa ROC - XGBoost")
plt.legend()
plt.grid(True)
plt.show()

# =========================================================
# 11. Ważność cech
# =========================================================
feature_importances = pd.DataFrame({
    "cecha": X_train.columns,
    "istotnosc": best_model.feature_importances_
}).sort_values(by="istotnosc", ascending=False)

print("\nNajważniejsze cechy:")
print(feature_importances)

# =========================================================
# 12. Learning curve XGBoost na podstawie xgb.cv
# =========================================================
dtrain = xgb.DMatrix(X_train, label=y_train)

cv_results = xgb.cv(
    params=best_model.get_xgb_params(),
    dtrain=dtrain,
    num_boost_round=50,
    nfold=5,
    metrics="auc",
    as_pandas=True,
    seed=42
)

plt.figure(figsize=(10, 6))
plt.plot(cv_results.index, cv_results["train-auc-mean"], label="train")
plt.plot(cv_results.index, cv_results["test-auc-mean"], label="test")
plt.xlabel("Number of trees")
plt.ylabel("Area under ROC curve")
plt.title("XGBoost learning curve")
plt.legend()
plt.grid(True)
plt.show()


Start programu...


FileNotFoundError: [Errno 2] No such file or directory: 'f1_processed_dataset.csv'